In [1]:
%load_ext autoreload
%autoreload 2

import os
if os.getcwd().split('/')[-1] != 'PreProcPipe':
    os.chdir('/teamspace/studios/this_studio/PreProcPipe')
os.getcwd()



'/teamspace/studios/this_studio/PreProcPipe'

在这个教程中我将以一个小数据集作为例子。

你可以看到我的repo中有[PreProcPipe/BraTS2021_Training_Data]()文件夹，里面有一些nii文件。

In [2]:

# load nii files
img_paths = []
root_dir = 'BraTS2021_Training_Data/BraTS2021_00000'
for folder in os.listdir(root_dir):
    folder_path = os.path.join(root_dir, folder)
    if os.path.isdir(folder_path):  # 检查是否为目录
        # 遍历子目录中的文件
        for file in os.listdir(folder_path):
            if file.endswith('.nii'):  # 检查是否为 .nii 文件
                img_paths.append(os.path.join(folder_path, file))  # 追加完整路径

img_paths


['BraTS2021_Training_Data/BraTS2021_00000/BraTS2021_00000_flair/00000057_brain_flair.nii',
 'BraTS2021_Training_Data/BraTS2021_00000/BraTS2021_00000_seg/00000057_final_seg.nii',
 'BraTS2021_Training_Data/BraTS2021_00000/BraTS2021_00000_t1/00000057_brain_t1.nii',
 'BraTS2021_Training_Data/BraTS2021_00000/BraTS2021_00000_t1ce/00000057_brain_t1ce.nii',
 'BraTS2021_Training_Data/BraTS2021_00000/BraTS2021_00000_t2/00000057_brain_t2.nii']

In [3]:
import nibabel as nib
import numpy as np

for img_path in img_paths:
    img = nib.load(img_path)
    img_data = img.get_fdata()
    print(img_data.shape)
    print(np.min(img_data), np.max(img_data))
    print()

(240, 240, 155)
0.0 2934.0

(240, 240, 155)
0.0 4.0

(240, 240, 155)
0.0 2023.0

(240, 240, 155)
0.0 12343.0

(240, 240, 155)
0.0 2421.0



可视化检查一下

In [4]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

# 从 img_paths 加载所有图像数据
def load_images(img_paths):
    images = []
    for path in img_paths:
        img = nib.load(path).get_fdata()  # 加载数据
        images.append(img)
    return images

# 显示某一 z 切片的函数
def show_slices(images, z):
    num_images = len(images)
    fig, axes = plt.subplots(1, num_images, figsize=(5 * num_images, 5))
    if num_images == 1:
        axes = [axes]

    for i, img in enumerate(images):
        axes[i].imshow(img[:, :, z], cmap="gray")
        axes[i].set_title(f"Image {i+1} - Z: {z}")
    plt.show()

images = load_images(img_paths)

# 使用 ipywidgets 创建滑块交互
z_max = images[0].shape[2] - 1
interact(lambda z: show_slices(images, z), z=IntSlider(min=0, max=z_max, step=1, value=z_max // 2))


interactive(children=(IntSlider(value=77, description='z', max=154), Output()), _dom_classes=('widget-interact…

<function __main__.<lambda>(z)>

万事俱备，我们开始使用`PreProcPipe`吧！

# PreProcPipe使用范例

为了将文件输入给PPP，我建议用一个像csv这样的元数据文件来记录文件路径，然后规范地输入。

很遗憾，这个制作csv的方法是因人而异的，因为每个数据集长得都不一样，不是吗😀

In [5]:
import os
import pandas as pd

def collect_metadata(root_dir):
    """
    Traverse the root_dir and collect the paths of multimodal images and segmentation for each sample.
    Return a list containing sample data, with each row corresponding to a sample and its file paths.
    """
    metadata = []
    for sample_folder in os.listdir(root_dir):
        sample_path = os.path.join(root_dir, sample_folder)
        if os.path.isdir(sample_path):  # Ensure it is a directory
            # Initialize a dictionary to store paths
            sample_data = {
                "sample_id": sample_folder,
                "t1": None,
                "t1ce": None,
                "t2": None,
                "flair": None,
                "seg": None
            }
            for modality_folder in os.listdir(sample_path):
                modality_path = os.path.join(sample_path, modality_folder)
                if os.path.isdir(modality_path):  # Ensure it is a modality subdirectory
                    for file in os.listdir(modality_path):
                        if file.endswith('.nii'):  # Ensure it is a .nii file
                            # Classify based on modality
                            if "t1.nii" in file and "ce" not in file:
                                sample_data["t1"] = os.path.join(modality_path, file)
                            elif "t1ce" in file:
                                sample_data["t1ce"] = os.path.join(modality_path, file)
                            elif "t2" in file:
                                sample_data["t2"] = os.path.join(modality_path, file)
                            elif "flair" in file:
                                sample_data["flair"] = os.path.join(modality_path, file)
                            elif "seg" in file:
                                sample_data["seg"] = os.path.join(modality_path, file)
            metadata.append(sample_data)
    return metadata

# Root directory path
root_dir = "BraTS2021_Training_Data"

# Collect metadata
metadata = collect_metadata(root_dir)

# Convert to DataFrame
df = pd.DataFrame(metadata)

# Save as CSV file
output_csv = f"{root_dir}/metadata.csv"
df.to_csv(output_csv, index=False)

print(f"Metadata saved to {output_csv}")

Metadata saved to BraTS2021_Training_Data/metadata.csv


In [7]:
import pandas as pd
from pipeline import SimplePreprocessor as ppp
from pipeline import run_in_parallel

# 定义读取 metadata.csv 并生成 cases 列表的函数
def load_cases_from_metadata(csv_path):
    """
    从 metadata.csv 加载病例信息，并生成 (image_paths, seg_path) 的列表。
    
    参数：
    - csv_path: metadata.csv 文件路径。
    
    返回：
    - cases: 包含病例信息的列表，每个元素是一个字典，格式为：
      {
          "sample_id": 样本ID,
          "image_paths": [模态1路径, 模态2路径, ...],
          "seg_path": 分割路径或 None
      }
    """
    df = pd.read_csv(csv_path)
    cases = []
    for _, row in df.iterrows():
        # 提取模态路径
        image_paths = [row['t1'], row['t1ce'], row['t2'], row['flair']]
        # 过滤掉空值
        image_paths = [path for path in image_paths if pd.notnull(path)]
        # 提取分割路径
        seg_path = row['seg'] if pd.notnull(row['seg']) else None
        # 添加到 cases
        cases.append({
            "sample_id": row['sample_id'],
            "image_paths": image_paths,
            "seg_path": seg_path
        })
    return cases

# 加载 metadata.csv
metadata_csv_path = "BraTS2021_Training_Data/metadata.csv"
cases = load_cases_from_metadata(metadata_csv_path)

# 初始化预处理器
preprocessor = ppp(
    target_spacing=[1.0, 1.0, 1.0], 
    normalization_scheme="z-score", 
    target_size=[256, 256]
)

# 使用多进程运行预处理
num_workers = 4  # 设置进程数
results = run_in_parallel(preprocessor, cases, num_workers=num_workers, output_root="preprocessed_data")



Step 1: Loading multi-modal image data...Step 1: Loading multi-modal image data...
Step 1: Loading multi-modal image data...


Step 1: Loading segmentation data...

Original image shape (modality 0): (240, 240, 155)
Original image shape (modality 1): (240, 240, 155)
Original image shape (modality 2): (240, 240, 155)
Original image shape (modality 3): (240, 240, 155)
Original segmentation shape: (240, 240, 155)

Step 2: Cropping to non-zero regions...

Step 1: Loading segmentation data...
Step 2: Cropping to non-zero regions along Z-axis...

Step 1: Loading segmentation data...
Original image shape (modality 0): (240, 240, 155)

Original image shape (modality 1): (240, 240, 155)Z-axis cropping range: 0 to 140

Original image shape (modality 2): (240, 240, 155)

Original image shape (modality 3): (240, 240, 155)
Shapes before cropping: [(240, 240, 155), (240, 240, 155), (240, 240, 155), (240, 240, 155)]Original image shape (modality 0): (240, 240, 155)
Original segmentation shape: (240, 



Step 4: Resampling data to target spacing...
Computed resize factors: [1.0, 1.0, 1.0]
Computed new shape: [240, 240, 140]
Resampling data...

Step 4: Resampling data to target spacing...
Computed resize factors: [1.0, 1.0, 1.0]
Computed new shape: [240, 240, 135]
Resampling data...

Step 4: Resampling data to target spacing...
Computed resize factors: [1.0, 1.0, 1.0]
Computed new shape: [240, 240, 146]
Resampling data...
Data resampled to shape: (240, 240, 140)
Resampling data...
Data resampled to shape: (240, 240, 135)
Resampling data...
Data resampled to shape: (240, 240, 146)
Resampling data...
Data resampled to shape: (240, 240, 140)
Resampling data...
Data resampled to shape: (240, 240, 135)

Resampling data...Data resampled to shape: (240, 240, 146)
Resampling data...
Data resampled to shape: (240, 240, 140)
Resampling data...
Data resampled to shape: (240, 240, 135)
Resampling data...
Data resampled to shape: (240, 240, 146)
Resampling data...
Data resampled to shape: (240, 24

In [8]:
results

[{'sample_id': 'BraTS2021_00000',
  'modality_paths': ['preprocessed_data/BraTS2021_00000/00000057_brain_t1.npz',
   'preprocessed_data/BraTS2021_00000/00000057_brain_t1ce.npz',
   'preprocessed_data/BraTS2021_00000/00000057_brain_t2.npz',
   'preprocessed_data/BraTS2021_00000/00000057_brain_flair.npz'],
  'seg_path': 'preprocessed_data/BraTS2021_00000/seg.npz',
  'meta_path': 'preprocessed_data/BraTS2021_00000/meta.npz'},
 {'sample_id': 'BraTS2021_00002',
  'modality_paths': ['preprocessed_data/BraTS2021_00002/00000014_brain_t1.npz',
   'preprocessed_data/BraTS2021_00002/00000014_brain_t1ce.npz',
   'preprocessed_data/BraTS2021_00002/00000014_brain_t2.npz',
   'preprocessed_data/BraTS2021_00002/00000014_brain_flair.npz'],
  'seg_path': 'preprocessed_data/BraTS2021_00002/seg.npz',
  'meta_path': 'preprocessed_data/BraTS2021_00002/meta.npz'},
 {'sample_id': 'BraTS2021_00003',
  'modality_paths': ['preprocessed_data/BraTS2021_00003/00000017_brain_t1.npz',
   'preprocessed_data/BraTS2021_

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import os

import pandas as pd
import ast
import os

metadata_csv = "preprocessed_data/metadata.csv"

# 目标 sample_id
target_sample_id = "BraTS2021_00002"

# 读取 CSV
df = pd.read_csv(metadata_csv)

# 查找目标样本行
row = df.loc[df['sample_id'] == target_sample_id].iloc[0]

# 解析 image_paths 列（它是一个字符串表示的列表）
image_paths = ast.literal_eval(row['modality_paths'])

# 分割路径
seg_path = row['seg_path'] if pd.notnull(row['seg_path']) else None

# 根据需要，也可以将这些路径与根目录拼接
# 如果 metadata.csv 中的路径已经是相对于 output_root 的相对路径
# 且 output_root 为 "preprocessed_data"
output_root = "preprocessed_data"
image_paths = [os.path.join(output_root, p) for p in image_paths]
if seg_path is not None:
    seg_path = os.path.join(output_root, seg_path)

# 此时，image_paths 和 seg_path 就是从 metadata 中获得的对应文件路径列表和分割路径
print("Modality paths:", image_paths)
print("Seg path:", seg_path)


# 假设所有文件都在 "preprocessed_data" 目录下
# image_paths = [os.path.join("preprocessed_data", p) for p in image_paths]
# seg_path = os.path.join("preprocessed_data", seg_path)

# 加载图像数据
modality_data = []
for path in image_paths:
    data = np.load(path)["data"] # npz 的 key 是data
    modality_data.append(data)

# 将模态合并为多通道数据 (H, W, D, C)
multi_modal_data = np.stack(modality_data, axis=-1)  # (H, W, D, C)

# 加载分割数据
seg_data = np.load(seg_path)["data"]  # (H, W, D)

# 获取数据形状和 D 轴大小
H, W, D, C = multi_modal_data.shape

def display_all_modalities(z_idx):
    """
    显示给定z轴索引下的所有模态图像及对应的分割mask叠加结果。
    """
    fig, axes = plt.subplots(1, C, figsize=(4*C, 4))
    
    # 遍历每个模态
    for i in range(C):
        img_slice = multi_modal_data[..., i][..., z_idx]
        print(f"min&max: {np.min(multi_modal_data[..., i]), np.max(multi_modal_data[..., i])}")  

        seg_slice = seg_data[..., z_idx]
        
        axes[i].imshow(img_slice, cmap='gray')
        
        # 使用 alpha 叠加 seg
        seg_mask = np.ma.masked_where(seg_slice == 0, seg_slice)
        axes[i].imshow(seg_mask, cmap='jet', alpha=0.5)
        
        axes[i].set_title(f"Modality {i}, Z={z_idx}")
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

# 使用交互式滑块：只需要控制 z_idx 即可
interact(
    display_all_modalities, 
    z_idx=IntSlider(min=0, max=D-1, step=1, value=D//2)
);


Modality paths: ['preprocessed_data/BraTS2021_00002/00000014_brain_t1.npz', 'preprocessed_data/BraTS2021_00002/00000014_brain_t1ce.npz', 'preprocessed_data/BraTS2021_00002/00000014_brain_t2.npz', 'preprocessed_data/BraTS2021_00002/00000014_brain_flair.npz']
Seg path: preprocessed_data/BraTS2021_00002/seg.npz


interactive(children=(IntSlider(value=67, description='z_idx', max=134), Output()), _dom_classes=('widget-inte…

大功告成噜💅